# Cascade sincnet → cnn_mel

Two thresholds on the sincnet score `s`:
- `s < α` → call it non-bird
- `s > β` → call it bird
- in between → ask cnn_mel and use its own threshold

For every budget X% (how often we're willing to wake mel up), find the (α, β) that gives the best test F2 while keeping `pct_mel ≤ X%`, and read off the metrics.

In [90]:
%load_ext autoreload
%autoreload 2

import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from utils import (
    SAMPLE_RATE,
    TARGET_AUDIO_LEN_TIME,
    make_time_datasets,
    make_mel_datasets,
    REPO_ROOT,
    MODELS_DIR,
    configure_tf_runtime,
    set_global_seed,
)

configure_tf_runtime()
set_global_seed()

SINC_TFLITE = MODELS_DIR / "sincnet_multi_tf.tflite"
MEL_TFLITE = MODELS_DIR / "cnn_mel_tf.tflite"
RECORDINGS_DIR = REPO_ROOT / "recordings"
RESULTS_DIR = REPO_ROOT / "results"

STEM = "cascade_sinctomel"

WINDOW_SAMPLES = TARGET_AUDIO_LEN_TIME  # 47872 = 3s @ 16kHz
STEP_SAMPLES = SAMPLE_RATE              # 1s step

for p in (SINC_TFLITE, MEL_TFLITE):
    if not p.exists():
        raise FileNotFoundError(f"missing model: {p}")
print(f"sincnet: {SINC_TFLITE}")
print(f"mel:     {MEL_TFLITE}")
print(f"window:  {WINDOW_SAMPLES} samples ({WINDOW_SAMPLES / SAMPLE_RATE:.2f} s), step {STEP_SAMPLES} samples")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
sincnet: /home/nathan/Documents/tiny-chirp-microflow/models/sincnet_multi_tf.tflite
mel:     /home/nathan/Documents/tiny-chirp-microflow/models/cnn_mel_tf.tflite
window:  47872 samples (2.99 s), step 16000 samples


Same inference loop as `evaluate_tflite_model` but it just returns the per-window score so we can do our own thresholding downstream.

In [91]:
class TFLiteRunner:
    def __init__(self, tflite_path: Path):
        self.interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
        self.interpreter.allocate_tensors()
        self.inp = self.interpreter.get_input_details()[0]
        self.out = self.interpreter.get_output_details()[0]
        in_scale, in_zp = self.inp["quantization"]
        out_scale, out_zp = self.out["quantization"]
        if in_scale == 0 or out_scale == 0:
            raise ValueError("tflite quantization scale is 0")
        self.in_scale, self.in_zp = in_scale, in_zp
        self.out_scale, self.out_zp = out_scale, out_zp
        self.qmin, self.qmax = (-128, 127) if self.inp["dtype"] == np.int8 else (0, 255)
        self.input_rank = len(self.inp["shape"])
        self.sample_rank = self.input_rank - 1

    def _adapt(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x)
        if x.ndim == self.input_rank or x.ndim == self.sample_rank:
            return x
        if self.sample_rank == 1 and x.ndim == 2 and x.shape[-1] == 1:
            return x.reshape(-1)
        if self.sample_rank == 3 and x.ndim == 1:
            return x[:, None, None]
        if self.sample_rank == 3 and x.ndim == 2 and x.shape[-1] == 1:
            return x[..., None]
        raise ValueError(f"unsupported sample rank: got {x.ndim}, expected {self.sample_rank}")

    def score(self, x: np.ndarray) -> float:
        x = self._adapt(x)
        x_q = np.clip(np.round(x / self.in_scale) + self.in_zp, self.qmin, self.qmax).astype(self.inp["dtype"])
        if x_q.ndim == self.sample_rank:
            x_q = x_q[None, ...]
        self.interpreter.set_tensor(self.inp["index"], x_q)
        self.interpreter.invoke()
        raw = ((self.interpreter.get_tensor(self.out["index"]).astype(np.float32) - self.out_zp) * self.out_scale).reshape(-1)
        if raw.size == 1:
            return float(1.0 / (1.0 + np.exp(-raw[0])))
        e = np.exp(raw - raw.max())
        return float(e[1] / e.sum())

sinc = TFLiteRunner(SINC_TFLITE)
mel = TFLiteRunner(MEL_TFLITE)
print(f"sincnet input shape: {sinc.inp['shape']}")
print(f"mel     input shape: {mel.inp['shape']}")

sincnet input shape: [    1 47872     1     1]
mel     input shape: [  1 184  80   1]


Grab the F2-best threshold each model picked during its own eval (the JSONL files), plus the test-set baselines we'll compare the cascade against.

In [92]:
def latest_record(jsonl_path: Path) -> dict:
    lines = jsonl_path.read_text().splitlines()
    if not lines:
        raise ValueError(f"empty results file: {jsonl_path}")
    return json.loads(lines[-1])

sinc_record = latest_record(RESULTS_DIR / "sincnet_multi_tf.jsonl")
mel_record = latest_record(RESULTS_DIR / "cnn_mel_tf.jsonl")

t_sinc = float(sinc_record["train"]["threshold"])
t_mel = float(mel_record["train"]["threshold"])
print(f"t_sinc = {t_sinc:.4f}")
print(f"t_mel  = {t_mel:.4f}")

# Single-model test baselines (used as horizontal references in the spec plots).
sinc_test_acc  = float(sinc_record["test"]["accuracy"])
sinc_test_prec = float(sinc_record["test"]["precision"])
sinc_test_rec  = float(sinc_record["test"]["recall"])
sinc_test_f2   = float(sinc_record["test"]["f2"])
mel_test_acc   = float(mel_record["test"]["accuracy"])
mel_test_prec  = float(mel_record["test"]["precision"])
mel_test_rec   = float(mel_record["test"]["recall"])
mel_test_f2    = float(mel_record["test"]["f2"])
print(f"  sincnet test:  acc={sinc_test_acc:.4f}  prec={sinc_test_prec:.4f}  rec={sinc_test_rec:.4f}  F2={sinc_test_f2:.4f}")
print(f"  cnn_mel test:  acc={mel_test_acc:.4f}  prec={mel_test_prec:.4f}  rec={mel_test_rec:.4f}  F2={mel_test_f2:.4f}")

t_sinc = 0.3320
t_mel  = 0.5904
  sincnet test:  acc=0.9598  prec=0.9020  rec=0.9847  F2=0.9670
  cnn_mel test:  acc=0.9777  prec=0.9402  rec=0.9956  F2=0.9840


Both dataset builders use `shuffle=False` on the test split, so zipping them gives aligned `(y, s_sinc, s_mel)` triples. The assert is there in case that ever stops being true.

In [93]:
def _adapt_for_sinc(x: np.ndarray) -> np.ndarray:
    # sincnet_multi expects rank-4 (T, 1, 1) per sample. make_time_datasets gives (T, 1).
    if x.ndim == 2 and x.shape[-1] == 1:
        return x[..., None]
    return x

_, _, test_time_ds, _ = make_time_datasets(batch_size=1)
_, _, test_mel_ds, _ = make_mel_datasets(batch_size=1)

y_true_list, s_sinc_list, s_mel_list = [], [], []

for (x_t, y_t), (x_m, y_m) in zip(test_time_ds.unbatch(), test_mel_ds.unbatch()):
    yt = int(y_t.numpy()); ym = int(y_m.numpy())
    if yt != ym:
        raise RuntimeError(f"label mismatch between time/mel pipelines: {yt} vs {ym}")
    y_true_list.append(yt)
    s_sinc_list.append(sinc.score(_adapt_for_sinc(x_t.numpy())))
    s_mel_list.append(mel.score(x_m.numpy()))

y_true = np.array(y_true_list, dtype=int)
s_sinc = np.array(s_sinc_list, dtype=float)
s_mel = np.array(s_mel_list, dtype=float)
print(f"scored {len(y_true)} test clips through both models")
print(f"  sinc score range: [{s_sinc.min():.3f}, {s_sinc.max():.3f}]")
print(f"  mel  score range: [{s_mel.min():.3f}, {s_mel.max():.3f}]")

Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
scored 1393 test clips through both models
  sinc score range: [0.000, 1.000]
  mel  score range: [0.000, 1.000]


2026-05-11 12:13:43.831336: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Run sincnet on every 3 s window of the recordings to see what the real score distribution looks like. Without this we have no idea what a "10% mel budget" actually means in the field.

In [94]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from tqdm.auto import tqdm

wavs = sorted(RECORDINGS_DIR.glob("*.wav"))
if not wavs:
    raise FileNotFoundError(f"No .wav files found in {RECORDINGS_DIR}.")

stream_scores = []
stream_meta = []
for wav in tqdm(wavs, desc="recordings", unit="file"):
    if wav.stat().st_size == 0:
        print(f"  skip {wav.name}: empty file")
        continue
    try:
        # soundfile tolerates non-strict RIFF (trailing bytes) that tf.audio.decode_wav rejects.
        audio, sr = sf.read(str(wav), dtype="float32", always_2d=True)
    except Exception as e:
        print(f"  skip {wav.name}: read failed ({e})")
        continue
    audio = audio.mean(axis=1)  # downmix N channels -> mono
    if sr != SAMPLE_RATE:
        g = gcd(sr, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // g, sr // g).astype(np.float32)
    n = len(audio)
    if n < WINDOW_SAMPLES:
        print(f"  skip {wav.name}: shorter than window after resample ({n} < {WINDOW_SAMPLES})")
        continue
    starts = range(0, n - WINDOW_SAMPLES + 1, STEP_SAMPLES)
    for start in tqdm(starts, desc=wav.name, unit="win", leave=False):
        win = audio[start:start + WINDOW_SAMPLES]
        stream_scores.append(sinc.score(win[..., None, None]))
        stream_meta.append((wav.name, start))

s_sinc_stream = np.array(stream_scores, dtype=float)
n_stream = len(s_sinc_stream)
if n_stream == 0:
    raise RuntimeError("recordings/ contained .wav files but no full 3s windows could be extracted")
print(f"scored {n_stream} stream windows from {len(set(m[0] for m in stream_meta))} recording(s)")
print(f"  sinc stream score range: [{s_sinc_stream.min():.3f}, {s_sinc_stream.max():.3f}]")

recordings:   0%|          | 0/34 [00:00<?, ?file/s]

R21_2022_02_16_07_00_01.wav:   0%|          | 0/518 [00:00<?, ?win/s]

R21_2022_02_16_07_17_15.wav:   0%|          | 0/897 [00:00<?, ?win/s]

R21_2022_02_16_07_32_17.wav:   0%|          | 0/745 [00:00<?, ?win/s]

R21_2022_02_16_08_16_03.wav:   0%|          | 0/897 [00:00<?, ?win/s]

...

R21_2022_02_22_09_37_54.wav:   0%|          | 0/292 [00:00<?, ?win/s]

R21_2022_02_22_09_43_52.wav:   0%|          | 0/897 [00:00<?, ?win/s]

R21_2022_02_23_07_23_44.wav:   0%|          | 0/897 [00:00<?, ?win/s]

R21_2022_02_23_07_38_46.wav:   0%|          | 0/897 [00:00<?, ?win/s]

scored 26251 stream windows from 34 recording(s)
  sinc stream score range: [0.000, 1.000]


## Pick a budget to highlight

Set `SELECTED_BUDGET_PCT` here; every plot and table below marks the F2-optimal operating point at this cap.

In [ ]:
SELECTED_BUDGET_PCT = 10.0  # change me

Threshold candidates are every unique sincnet score plus the midpoints between them. Then compute every metric across the full (α, β) grid in one shot using prefix sums.

In [ ]:
# SELECT A %

In [95]:
# --- candidate thresholds: unique scores ∪ midpoints between them ---
unique_s = np.unique(np.concatenate([s_sinc, s_sinc_stream]))
midpoints = (unique_s[1:] + unique_s[:-1]) / 2.0
cands = np.sort(np.concatenate([unique_s, midpoints]))
L = len(cands)
print(f"{L} candidate thresholds ({len(unique_s)} unique scores + {len(midpoints)} midpoints)")

# --- sort test data, build prefix sums for O(1) range queries ---
order_test = np.argsort(s_sinc, kind="mergesort")
s_test_sorted = s_sinc[order_test]
y_test_sorted = y_true[order_test].astype(np.int64)
mel_pred_full = (s_mel >= t_mel).astype(np.int64)
m_test_sorted = mel_pred_full[order_test]
ym_test_sorted = y_test_sorted * m_test_sorted

y_cum  = np.concatenate(([0], np.cumsum(y_test_sorted))).astype(np.int64)
m_cum  = np.concatenate(([0], np.cumsum(m_test_sorted))).astype(np.int64)
ym_cum = np.concatenate(([0], np.cumsum(ym_test_sorted))).astype(np.int64)
N_test = len(s_sinc)
P_pos = int(y_cum[-1])

s_stream_sorted = np.sort(s_sinc_stream)
N_stream = len(s_sinc_stream)

# searchsorted(side="left")  -> # clips with s < c
# searchsorted(side="right") -> # clips with s ≤ c
IA_test   = np.searchsorted(s_test_sorted,   cands, side="left").astype(np.int64)
IB_test   = np.searchsorted(s_test_sorted,   cands, side="right").astype(np.int64)
IA_stream = np.searchsorted(s_stream_sorted, cands, side="left").astype(np.int64)
IB_stream = np.searchsorted(s_stream_sorted, cands, side="right").astype(np.int64)

# Broadcast: rows = α index, cols = β index.
A   = IA_test[:, None]
B   = IB_test[None, :]
A_s = IA_stream[:, None]
B_s = IB_stream[None, :]

# Band (α ≤ s ≤ β) on test set.
yc_diff  = y_cum[B]  - y_cum[A]   # # positives in band
mc_diff  = m_cum[B]  - m_cum[A]   # # mel-positive in band
ymc_diff = ym_cum[B] - ym_cum[A]  # # (positive AND mel-positive) in band
n_band   = B - A
TP_band = ymc_diff
FP_band = mc_diff - ymc_diff
FN_band = yc_diff - ymc_diff
TN_band = n_band - TP_band - FP_band - FN_band

# Below band → predict 0.
FN_below = y_cum[A]
TN_below = A - FN_below

# Above band → predict 1.
TP_above = P_pos - y_cum[B]
FP_above = (N_test - B) - TP_above

TP = (TP_band + TP_above).astype(np.int64)
FP = (FP_band + FP_above).astype(np.int64)
FN = (FN_below + FN_band).astype(np.int64)
TN = (TN_below + TN_band).astype(np.int64)

# Metrics (safe-divide, zero on empty denominators).
TP_f = TP.astype(np.float64)
denom_p = (TP + FP).astype(np.float64)
denom_r = (TP + FN).astype(np.float64)
prec_mat = np.divide(TP_f, denom_p, out=np.zeros_like(TP_f), where=denom_p > 0)
rec_mat  = np.divide(TP_f, denom_r, out=np.zeros_like(TP_f), where=denom_r > 0)
f2_denom = 4.0 * prec_mat + rec_mat
f2_mat = np.divide(5.0 * prec_mat * rec_mat, f2_denom, out=np.zeros_like(prec_mat), where=f2_denom > 0)
acc_mat = (TP + TN).astype(np.float64) / N_test

# Energy budget = mel-trigger rate on recordings stream.
pct_mat = (B_s - A_s).astype(np.float64) / N_stream * 100.0

# Invalid pairs: α > β.
valid_mat = np.arange(L)[:, None] <= np.arange(L)[None, :]

print(f"matrices shape: {f2_mat.shape}")
print(f"valid pairs: {int(valid_mat.sum())}")
print(f"pct range: [{pct_mat[valid_mat].min():.2f}%, {pct_mat[valid_mat].max():.2f}%]")
print(f"F2  range: [{f2_mat[valid_mat].min():.4f}, {f2_mat[valid_mat].max():.4f}]")

309 candidate thresholds (155 unique scores + 154 midpoints)
matrices shape: (309, 309)
valid pairs: 47895
pct range: [0.00%, 100.00%]
F2  range: [0.0379, 0.9883]


In [ ]:
# Selected operating point: F2-optimal (α, β) with pct_mel ≤ SELECTED_BUDGET_PCT.
_pct_v = np.where(valid_mat, pct_mat, np.inf)
_f2_v  = np.where(valid_mat & (_pct_v <= SELECTED_BUDGET_PCT), f2_mat, -np.inf)
_flat = int(np.argmax(_f2_v))
sel_a_idx, sel_b_idx = np.unravel_index(_flat, f2_mat.shape)
sel_alpha = float(cands[sel_a_idx])
sel_beta  = float(cands[sel_b_idx])
sel_pct   = float(pct_mat[sel_a_idx, sel_b_idx])
sel_f2    = float(f2_mat[sel_a_idx, sel_b_idx])
sel_acc   = float(acc_mat[sel_a_idx, sel_b_idx])
sel_prec  = float(prec_mat[sel_a_idx, sel_b_idx])
sel_rec   = float(rec_mat[sel_a_idx, sel_b_idx])
print(f"selected @ budget≤{SELECTED_BUDGET_PCT:.1f}%: alpha={sel_alpha:.4f}  beta={sel_beta:.4f}  "
      f"pct={sel_pct:.2f}%  F2={sel_f2:.4f}  acc={sel_acc:.4f}  prec={sel_prec:.4f}  rec={sel_rec:.4f}")

def _heat(ax, mat, title, cmap, vmin=None, vmax=None):
    # rows of mat = α index, cols = β index. Transpose so x = α, y = β.
    m = np.where(valid_mat, mat, np.nan).T
    im = ax.imshow(m, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax,
                   extent=[cands[0], cands[-1], cands[0], cands[-1]],
                   aspect="auto", interpolation="nearest")
    ax.plot(sel_alpha, sel_beta, marker="x", color="cyan",
            markersize=11, markeredgewidth=2.0)
    ax.set_xlabel("α")
    ax.set_ylabel("β")
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.85)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
_heat(axes[0, 0], pct_mat,  "pct_mel (% stream → mel)", "viridis", 0, 100)
_heat(axes[0, 1], f2_mat,   "F2 (test)",                "magma")
_heat(axes[0, 2], acc_mat,  "Accuracy (test)",          "magma")
_heat(axes[1, 0], prec_mat, "Precision (test)",         "magma")
_heat(axes[1, 1], rec_mat,  "Recall (test)",            "magma")
axes[1, 2].axis("off")
fig.suptitle(f"Cyan × = F2-optimal (α, β) at budget ≤ {SELECTED_BUDGET_PCT:.1f}%", fontsize=10)
plt.tight_layout()
plt.show()

Pick the (α, β) that gives the highest F2 you can buy with at most X% of windows going to mel. Sweep X and keep the chosen pair and its metrics.

In [ ]:
budgets = np.linspace(0.0, 100.0, 401)

f2_valid  = np.where(valid_mat, f2_mat,  -np.inf)
pct_valid = np.where(valid_mat, pct_mat, np.inf)

best_alpha = np.empty(len(budgets))
best_beta  = np.empty(len(budgets))
best_pct   = np.empty(len(budgets))
best_f2    = np.empty(len(budgets))
best_acc   = np.empty(len(budgets))
best_prec  = np.empty(len(budgets))
best_rec   = np.empty(len(budgets))

for i, x in enumerate(budgets):
    feasible = pct_valid <= x
    f2_masked = np.where(feasible, f2_valid, -np.inf)
    flat = int(np.argmax(f2_masked))
    a_idx, b_idx = np.unravel_index(flat, f2_mat.shape)
    best_alpha[i] = cands[a_idx]
    best_beta[i]  = cands[b_idx]
    best_pct[i]   = pct_mat[a_idx, b_idx]
    best_f2[i]    = f2_mat[a_idx, b_idx]
    best_acc[i]   = acc_mat[a_idx, b_idx]
    best_prec[i]  = prec_mat[a_idx, b_idx]
    best_rec[i]   = rec_mat[a_idx, b_idx]

assert np.all(np.diff(best_f2) >= -1e-12), "best_f2(budget) should be non-decreasing"

print(f"budget=0%:   alpha={best_alpha[0]:.4f}  beta={best_beta[0]:.4f}  pct={best_pct[0]:.2f}%  "
      f"F2={best_f2[0]:.4f}  (sincnet alone: {sinc_test_f2:.4f})")
print(f"budget=100%: alpha={best_alpha[-1]:.4f}  beta={best_beta[-1]:.4f}  pct={best_pct[-1]:.2f}%  "
      f"F2={best_f2[-1]:.4f}  (cnn_mel alone: {mel_test_f2:.4f})")

import pandas as pd
snap_budgets = [0.0, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0]
snap_rows = []
# Lead with the user-selected budget so it's easy to spot.
snap_rows.append({
    "budget %":  SELECTED_BUDGET_PCT,
    "alpha":     sel_alpha,
    "beta":      sel_beta,
    "pct_mel %": sel_pct,
    "F2":        sel_f2,
    "accuracy":  sel_acc,
    "precision": sel_prec,
    "recall":    sel_rec,
    "tag":       "← SELECTED",
})
for x in snap_budgets:
    feasible = pct_valid <= x
    f2_masked = np.where(feasible, f2_valid, -np.inf)
    flat = int(np.argmax(f2_masked))
    a_idx, b_idx = np.unravel_index(flat, f2_mat.shape)
    snap_rows.append({
        "budget %":  x,
        "alpha":     cands[a_idx],
        "beta":      cands[b_idx],
        "pct_mel %": pct_mat[a_idx, b_idx],
        "F2":        f2_mat[a_idx, b_idx],
        "accuracy":  acc_mat[a_idx, b_idx],
        "precision": prec_mat[a_idx, b_idx],
        "recall":    rec_mat[a_idx, b_idx],
        "tag":       "",
    })
pd.DataFrame(snap_rows).style.format({
    "budget %":  "{:.1f}",
    "alpha":     "{:.4f}",
    "beta":      "{:.4f}",
    "pct_mel %": "{:.2f}",
    "F2":        "{:.4f}",
    "accuracy":  "{:.4f}",
    "precision": "{:.4f}",
    "recall":    "{:.4f}",
})

What the cascade actually buys you at each budget.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

panels = [
    (axes[0, 0], best_f2,   "F2",        sinc_test_f2,   mel_test_f2,   sel_f2),
    (axes[0, 1], best_acc,  "Accuracy",  sinc_test_acc,  mel_test_acc,  sel_acc),
    (axes[1, 0], best_prec, "Precision", sinc_test_prec, mel_test_prec, sel_prec),
    (axes[1, 1], best_rec,  "Recall",    sinc_test_rec,  mel_test_rec,  sel_rec),
]
for ax, y_curve, y_label, sinc_ref, mel_ref, sel_y in panels:
    ax.plot(budgets, y_curve, linewidth=2, label="F2-optimal cascade")
    ax.axhline(sinc_ref, linestyle=":", color="tab:orange", linewidth=1.2, label=f"sincnet alone ({sinc_ref:.4f})")
    ax.axhline(mel_ref,  linestyle=":", color="tab:green",  linewidth=1.2, label=f"cnn_mel alone ({mel_ref:.4f})")
    ax.axvline(SELECTED_BUDGET_PCT, linestyle="--", color="cyan", linewidth=1.2,
               label=f"selected ({SELECTED_BUDGET_PCT:.1f}%)")
    ax.plot(SELECTED_BUDGET_PCT, sel_y, marker="x", color="cyan", markersize=11, markeredgewidth=2)
    y_min = float(min(np.nanmin(y_curve), sinc_ref, mel_ref))
    pad = max(1e-4, (1.0 - y_min) * 0.05)
    ax.set_ylim(y_min - pad, 1.0)
    ax.set_xlabel("budget cap: max % of recording windows triggering mel")
    ax.set_ylabel(y_label)
    ax.set_title(f"{y_label} at F2-optimal (α, β) under budget cap")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(1, 1, figsize=(8, 4))
ax2.plot(budgets, best_alpha, label="α (low)",  color="tab:red")
ax2.plot(budgets, best_beta,  label="β (high)", color="tab:blue")
ax2.fill_between(budgets, best_alpha, best_beta, alpha=0.15, color="tab:purple", label="band → mel")
ax2.axvline(SELECTED_BUDGET_PCT, linestyle="--", color="cyan", linewidth=1.2,
            label=f"selected ({SELECTED_BUDGET_PCT:.1f}%)")
ax2.plot(SELECTED_BUDGET_PCT, sel_alpha, marker="x", color="cyan", markersize=11, markeredgewidth=2)
ax2.plot(SELECTED_BUDGET_PCT, sel_beta,  marker="x", color="cyan", markersize=11, markeredgewidth=2)
ax2.set_xlabel("budget cap (% mel)")
ax2.set_ylabel("sincnet score")
ax2.set_title("F2-optimal (α, β) vs budget cap")
ax2.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
ax2.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

Dump to disk so we don't re-run all this next time.

In [99]:
out_path = RESULTS_DIR / "cascade_sincnet_multi__cnn_mel.json"
payload = {
    "stage1": "sincnet_multi",
    "stage2": "cnn_mel",
    "t_mel": t_mel,
    "n_stream_windows": int(N_stream),
    "n_test_clips": int(N_test),
    "n_candidates": int(L),
    "budgets": budgets.tolist(),
    "best_by_f2_under_budget": {
        "alpha":     best_alpha.tolist(),
        "beta":      best_beta.tolist(),
        "pct_mel":   best_pct.tolist(),
        "f2":        best_f2.tolist(),
        "accuracy":  best_acc.tolist(),
        "precision": best_prec.tolist(),
        "recall":    best_rec.tolist(),
    },
}
out_path.write_text(json.dumps(payload, indent=2))
print(f"wrote {out_path}")

wrote /home/nathan/Documents/tiny-chirp-microflow/results/cascade_sincnet_multi__cnn_mel.json


## Inference time vs budget

Plug in the measured per-window inference times below. The cascade always runs sincnet, and mel fires on the chosen band — at rate `best_pct[X] / 100` for the F2-optimal (α, β) at each budget cap.

```
cascade_time(X) = SINCNET_AVERAGE + (best_pct[X] / 100) * MEL_CNN_AVERAGE
```

In [ ]:
# Per-window inference time in ms. Fill in once measured.
SINCNET_AVERAGE = 1.0     # ms
MEL_CNN_AVERAGE = 5.0     # ms

cascade_time = SINCNET_AVERAGE + (best_pct / 100.0) * MEL_CNN_AVERAGE
sel_time = SINCNET_AVERAGE + (sel_pct / 100.0) * MEL_CNN_AVERAGE

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(budgets, cascade_time, linewidth=2, label="cascade (sinc + pct·mel)")
ax.axhline(SINCNET_AVERAGE, linestyle=":", color="tab:orange", linewidth=1.2,
           label=f"sincnet alone ({SINCNET_AVERAGE:.2f} ms)")
ax.axhline(MEL_CNN_AVERAGE, linestyle=":", color="tab:green", linewidth=1.2,
           label=f"cnn_mel alone ({MEL_CNN_AVERAGE:.2f} ms)")
ax.axvline(SELECTED_BUDGET_PCT, linestyle="--", color="cyan", linewidth=1.2,
           label=f"selected ({SELECTED_BUDGET_PCT:.1f}%)")
ax.plot(SELECTED_BUDGET_PCT, sel_time, marker="x", color="cyan", markersize=11, markeredgewidth=2)
ax.set_xlabel("budget cap (% mel)")
ax.set_ylabel("avg time per window (ms)")
ax.set_title("Cascade inference time at the F2-optimal (α, β)")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
ax.legend(fontsize=8, loc="best")

ax = axes[1]
ax.plot(budgets, best_pct, linewidth=2, label="actual pct_mel of chosen (α, β)")
ax.plot([0, 100], [0, 100], linestyle="--", color="gray", linewidth=1, label="y = x (cap)")
ax.axvline(SELECTED_BUDGET_PCT, linestyle="--", color="cyan", linewidth=1.2,
           label=f"selected ({SELECTED_BUDGET_PCT:.1f}%)")
ax.plot(SELECTED_BUDGET_PCT, sel_pct, marker="x", color="cyan", markersize=11, markeredgewidth=2)
ax.set_xlabel("budget cap (% mel)")
ax.set_ylabel("actual % windows triggering mel")
ax.set_title("Budget cap vs realised mel usage")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
ax.legend(fontsize=8, loc="best")

plt.tight_layout()
plt.show()

print(f"budget=0%:     time={cascade_time[0]:.3f} ms  pct_mel={best_pct[0]:.2f}%  (sincnet alone: {SINCNET_AVERAGE:.3f} ms)")
print(f"budget={SELECTED_BUDGET_PCT:.1f}%:  time={sel_time:.3f} ms  pct_mel={sel_pct:.2f}%  ← SELECTED")
print(f"budget=100%:   time={cascade_time[-1]:.3f} ms  pct_mel={best_pct[-1]:.2f}%  (cnn_mel alone: {MEL_CNN_AVERAGE:.3f} ms)")